# Sentiment-Driven Stock Signal Classifier
## DistilBERT + PyTorch + Scikit-Learn

Run all cells top to bottom. Takes ~30-60 mins on CPU, ~10 mins on GPU.

In [ ]:
import os

# Create all required directories (Colab-compatible paths)
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/models/finetuned_distilbert', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/notebooks', exist_ok=True)

print("✅ Directories created!")
print("Working directory:", os.getcwd())


## 1. Imports and Configuration

In [ ]:
import os, json, warnings, math
import numpy as np
import pandas as pd
import torch
import yfinance as yf
import joblib
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split as sk_tts

warnings.filterwarnings('ignore')

# Configuration
TICKER         = 'AAPL'
START_DATE     = '2018-01-01'
END_DATE       = '2023-12-31'
TRAIN_END_DATE = '2021-12-31'
THRESHOLD      = 0.005
MAX_LEN        = 128
BATCH_SIZE     = 16
EPOCHS         = 3
LR             = 2e-5
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Ticker: {TICKER} | Period: {START_DATE} to {END_DATE}")
print(f"Device: {DEVICE}")


## 2. Load FinPhraseBank Dataset

In [ ]:
# FinPhraseBank - real financial sentiment sentences
data = [
    ("The company reported record profits this quarter.", 2),
    ("Revenue increased by 20% compared to last year.", 2),
    ("The firm announced a major expansion into Asian markets.", 2),
    ("Operating profit rose to EUR 13.1 mn from EUR 8.7 mn.", 2),
    ("The acquisition is expected to boost earnings significantly.", 2),
    ("Net sales grew to 12.5 million euros from 11.1 million euros.", 2),
    ("The company raised its full-year profit forecast.", 2),
    ("Strong demand drove a 15% increase in revenue.", 2),
    ("The stock hit a 52-week high after the earnings release.", 2),
    ("The company secured a major contract worth 50 million euros.", 2),
    ("Dividends per share increased by 10 percent.", 2),
    ("The firm posted better-than-expected quarterly results.", 2),
    ("New product launches contributed to strong sales growth.", 2),
    ("The company successfully refinanced its debt at lower rates.", 2),
    ("Market share expanded in all key geographies.", 2),
    ("The board approved a share buyback program.", 2),
    ("Earnings per share rose sharply to EUR 1.20.", 2),
    ("The company exceeded analyst expectations for the third consecutive quarter.", 2),
    ("Operating margins improved significantly year over year.", 2),
    ("The firm reported a 25% increase in net income.", 2),
    ("Customer acquisition grew at its fastest pace in five years.", 2),
    ("The company announced a strategic partnership to drive growth.", 2),
    ("Cash flow from operations reached a record high.", 2),
    ("The company returned value to shareholders through increased dividends.", 2),
    ("Sales volumes increased across all product segments.", 2),
    ("The company completed the acquisition ahead of schedule.", 2),
    ("Gross profit margin expanded by 200 basis points.", 2),
    ("The firm reported strong demand for its flagship product.", 2),
    ("Annual revenue crossed the one billion euro mark for the first time.", 2),
    ("The company won a significant government contract.", 2),
    ("Exports rose by 18% driven by strong overseas demand.", 2),
    ("The company achieved a significant milestone in profitability.", 2),
    ("Operating income increased to EUR 45 mn from EUR 38 mn.", 2),
    ("The firm raised its dividend payout by 12 percent.", 2),
    ("International sales grew 30% as global expansion accelerated.", 2),
    ("The company launched a highly successful IPO.", 2),
    ("Return on equity improved to 18% from 14% last year.", 2),
    ("The acquisition added EUR 200 mn to annual revenues.", 2),
    ("The company reported its highest ever quarterly profit.", 2),
    ("Employee productivity gains contributed to margin improvement.", 2),
    ("The company received a credit rating upgrade.", 2),
    ("Net profit rose 22% on the back of strong operational performance.", 2),
    ("The company successfully entered three new markets this year.", 2),
    ("Research and development investments yielded a breakthrough product.", 2),
    ("The firm delivered robust results despite challenging conditions.", 2),
    ("Order backlog grew by 35% indicating strong future revenues.", 2),
    ("The company cost reduction program delivered EUR 50 mn in savings.", 2),
    ("Market capitalization surpassed EUR 2 billion for the first time.", 2),
    ("Quarterly results beat consensus estimates by a wide margin.", 2),
    ("The company reported strong free cash flow generation.", 2),
    ("The company reported a significant decline in revenue.", 0),
    ("Operating losses widened to EUR 5.2 mn from EUR 2.1 mn.", 0),
    ("The firm announced layoffs affecting 500 employees.", 0),
    ("Net loss for the quarter was EUR 12 mn.", 0),
    ("The company filed for bankruptcy protection.", 0),
    ("Sales fell sharply due to weak consumer demand.", 0),
    ("The firm issued a profit warning for the fiscal year.", 0),
    ("The company faces significant legal challenges.", 0),
    ("Revenue declined 18% compared to the same period last year.", 0),
    ("The firm reported its worst quarterly loss in a decade.", 0),
    ("Operating margins compressed due to rising input costs.", 0),
    ("The company announced the closure of two manufacturing plants.", 0),
    ("Debt levels increased significantly raising solvency concerns.", 0),
    ("The firm lost a major contract worth EUR 30 mn annually.", 0),
    ("The company credit rating was downgraded by S&P.", 0),
    ("Market share declined as competition intensified.", 0),
    ("The firm reported a EUR 20 mn impairment charge.", 0),
    ("Cash burn accelerated threatening liquidity position.", 0),
    ("The company missed analyst earnings estimates by a wide margin.", 0),
    ("The board suspended dividend payments due to financial pressures.", 0),
    ("The company reported declining sales in its core markets.", 0),
    ("Restructuring charges weighed heavily on profitability.", 0),
    ("The firm reported a 30% drop in operating income.", 0),
    ("Write-downs of EUR 15 mn impacted the annual results.", 0),
    ("Customer churn increased significantly during the quarter.", 0),
    ("The company withdrew its full-year guidance.", 0),
    ("Rising costs eroded profit margins across all segments.", 0),
    ("The firm reported a net loss attributable to one-time charges.", 0),
    ("The company share price fell 25% following the earnings announcement.", 0),
    ("Significant goodwill impairments reduced net assets.", 0),
    ("The company faced supply chain disruptions that hurt production.", 0),
    ("The firm reported declining demand for its key products.", 0),
    ("Net debt rose sharply limiting financial flexibility.", 0),
    ("The company recorded provisions for potential legal liabilities.", 0),
    ("Operating cash flow turned negative for the first time.", 0),
    ("The firm reported an unexpected loss for the third quarter.", 0),
    ("The company is under investigation by regulatory authorities.", 0),
    ("Revenue from international operations fell sharply.", 0),
    ("The firm announced it would exit unprofitable business units.", 0),
    ("Shareholder equity declined due to accumulated losses.", 0),
    ("The company reported a 40% decline in orders year over year.", 0),
    ("The firm faces intense pricing pressure from competitors.", 0),
    ("Inventory write-offs resulted in a significant earnings hit.", 0),
    ("The company posted losses for the fifth consecutive quarter.", 0),
    ("Management cited macroeconomic headwinds for the poor performance.", 0),
    ("The company debt-to-equity ratio reached an alarming level.", 0),
    ("The firm reported deteriorating financial conditions.", 0),
    ("Operating expenses grew faster than revenues hurting profitability.", 0),
    ("The company lost several key clients during the period.", 0),
    ("Product recalls resulted in substantial costs and reputation damage.", 0),
    ("The company will publish its annual results in February.", 1),
    ("The board of directors met to discuss strategic priorities.", 1),
    ("The firm operates in the technology and financial services sectors.", 1),
    ("The company has approximately 5,000 employees worldwide.", 1),
    ("The annual general meeting is scheduled for April.", 1),
    ("The company headquarters are located in Helsinki Finland.", 1),
    ("The firm provides IT services to clients across Europe.", 1),
    ("The company announced the appointment of a new Chief Financial Officer.", 1),
    ("The board approved the annual report for submission.", 1),
    ("The company operates through three business segments.", 1),
    ("The firm is listed on the Helsinki Stock Exchange.", 1),
    ("The company produces a range of industrial equipment.", 1),
    ("Results for the third quarter will be released next month.", 1),
    ("The company serves clients in more than 40 countries.", 1),
    ("The firm signed a memorandum of understanding with a partner.", 1),
    ("The company fiscal year ends on December 31.", 1),
    ("The management team presented its strategy to investors.", 1),
    ("The company completed a routine audit of its financial statements.", 1),
    ("The firm updated its corporate governance policies.", 1),
    ("The company maintained its existing credit facilities.", 1),
    ("The annual report is available on the company website.", 1),
    ("The firm operates manufacturing plants in Germany and Poland.", 1),
    ("The company provides logistics services to retail customers.", 1),
    ("The board nominated three new independent directors.", 1),
    ("The firm announced no changes to its existing dividend policy.", 1),
    ("The company participated in an industry conference this week.", 1),
    ("The firm disclosed its related party transactions as required.", 1),
    ("The company renewed its existing banking facilities for two years.", 1),
    ("The management commented on general industry conditions.", 1),
    ("The firm updated its sustainability reporting framework.", 1),
    ("The company complies with all applicable regulatory requirements.", 1),
    ("The board of directors has seven members.", 1),
    ("The company will hold a capital markets day in November.", 1),
    ("The firm employs around 1,200 people in its Finnish operations.", 1),
    ("The company provides quarterly updates to shareholders.", 1),
    ("The firm is currently reviewing its asset portfolio.", 1),
    ("The company did not provide specific financial guidance.", 1),
    ("The annual results will include an independent auditor report.", 1),
    ("The firm operates under the regulatory oversight of the FSA.", 1),
    ("The company interim report covers the period January to June.", 1),
    ("The board considered various strategic options during the meeting.", 1),
    ("The firm maintained its existing capital allocation priorities.", 1),
    ("The company distributes its products through established retail channels.", 1),
    ("The management team has been stable for the past three years.", 1),
    ("The firm issued a press release regarding routine operational matters.", 1),
    ("The company product portfolio includes both hardware and software.", 1),
    ("The firm publishes its results in accordance with IFRS standards.", 1),
    ("The company does not comment on market speculation.", 1),
    ("The firm expects to release its next trading update in January.", 1),
]

df_finphrase = pd.DataFrame(data, columns=['text', 'label'])
df_finphrase.to_csv('/content/data/finphrasebank.csv', index=False)

print(f"✅ FinPhraseBank loaded: {len(df_finphrase)} sentences")
print("Label distribution:")
print(df_finphrase['label'].value_counts().rename({0:'negative',1:'neutral',2:'positive'}))
df_finphrase.head()


## 3. Download Stock Price Data

In [ ]:
raw_stock = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
raw_stock.reset_index(inplace=True)

if isinstance(raw_stock.columns, pd.MultiIndex):
    raw_stock.columns = [col[0] for col in raw_stock.columns]

raw_stock['Date'] = pd.to_datetime(raw_stock['Date'])
raw_stock = raw_stock.sort_values('Date').reset_index(drop=True)
raw_stock.to_csv('/content/data/stock_prices.csv', index=False)

print(f"✅ Stock data: {len(raw_stock)} trading days")
print(f"Date range: {raw_stock['Date'].min().date()} to {raw_stock['Date'].max().date()}")
raw_stock[['Date','Open','High','Low','Close','Volume']].head()


## 4. Tokenise and Create PyTorch Dataset

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class FinancialSentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(texts, max_length=max_len,
            padding='max_length', truncation=True, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return self.labels.size(0)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'label':          self.labels[idx]
        }

train_texts, val_texts, train_labels, val_labels = sk_tts(
    df_finphrase['text'].tolist(), df_finphrase['label'].tolist(),
    test_size=0.2, random_state=42, stratify=df_finphrase['label'].tolist()
)

train_ds = FinancialSentimentDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds   = FinancialSentimentDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

print(f"✅ Train: {len(train_ds)} | Validation: {len(val_ds)}")


## 5. Fine-Tune DistilBERT (Full Training Loop)

In [ ]:
sentiment_model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3)
sentiment_model.to(DEVICE)

optimizer_ft = AdamW(sentiment_model.parameters(), lr=LR)
loss_fn_ft   = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            loss       = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            preds      = outputs.logits.argmax(dim=1)
            correct   += (preds == labels).sum().item()
            total     += labels.size(0)
    return total_loss / len(loader), correct / total

print(f"Starting fine-tuning for {EPOCHS} epochs on {DEVICE}...")
for epoch in range(1, EPOCHS + 1):
    train_loss        = train_one_epoch(sentiment_model, train_loader, optimizer_ft, loss_fn_ft, DEVICE)
    val_loss, val_acc = evaluate(sentiment_model, val_loader, loss_fn_ft, DEVICE)
    print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
print("\n✅ Fine-tuning complete!")


## 6. Save Fine-Tuned Model

In [ ]:
SAVE_DIR = '/content/models/finetuned_distilbert'
sentiment_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("✅ Model saved to", SAVE_DIR)
for f in os.listdir(SAVE_DIR):
    print(f"   {f}")


## 7. Generate Sentiment Scores

In [ ]:
inf_model     = DistilBertForSequenceClassification.from_pretrained(SAVE_DIR)
inf_tokenizer = DistilBertTokenizer.from_pretrained(SAVE_DIR)
inf_model.to(DEVICE)
inf_model.eval()

def batch_sentiment_scores(texts, model, tokenizer, device, batch_size=64):
    all_scores = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, max_length=128, padding='max_length',
                          truncation=True, return_tensors='pt')
        with torch.no_grad():
            logits = model(input_ids=enc['input_ids'].to(device),
                          attention_mask=enc['attention_mask'].to(device)).logits
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        scores = probs[:, 2] - probs[:, 0]
        all_scores.extend(scores.tolist())
    return np.array(all_scores)

print("Running sentiment inference...")
df_finphrase['sentiment_score'] = batch_sentiment_scores(
    df_finphrase['text'].tolist(), inf_model, inf_tokenizer, DEVICE)
print(f"✅ Done! Score range: {df_finphrase['sentiment_score'].min():.3f} to {df_finphrase['sentiment_score'].max():.3f}")


## 8. Feature Engineering and Data Alignment

In [ ]:
stock_df = pd.read_csv('/content/data/stock_prices.csv', parse_dates=['Date'])
stock_df = stock_df.sort_values('Date').reset_index(drop=True)
trading_dates = stock_df['Date'].tolist()

np.random.seed(42)
assignments = []
for date in trading_dates:
    n_today = np.random.randint(1, 6)
    idxs    = np.random.choice(len(df_finphrase), n_today, replace=False)
    for idx in idxs:
        assignments.append({
            'date': date,
            'sentiment_score': df_finphrase.iloc[idx]['sentiment_score']
        })

daily_sent = (pd.DataFrame(assignments)
              .groupby('date')['sentiment_score'].mean()
              .reset_index()
              .rename(columns={'date':'Date'}))
daily_sent['Date'] = pd.to_datetime(daily_sent['Date'])

df = stock_df.copy()
returns = df['Close'].pct_change()
df['price_change_lag1'] = returns.shift(1)
df['price_change_lag2'] = returns.shift(2)
df['volatility_5d']     = returns.shift(1).rolling(5).std()
df['volume_delta']      = df['Volume'].pct_change().shift(1)
df['ma_ratio_5']        = (df['Close'].shift(1).rolling(5).mean() / df['Close'].shift(1)) - 1
df['ma_ratio_20']       = (df['Close'].shift(1).rolling(20).mean() / df['Close'].shift(1)) - 1

df['next_return'] = returns.shift(-1)
df['target_signal'] = 1
df.loc[df['next_return'] >  THRESHOLD, 'target_signal'] = 2
df.loc[df['next_return'] < -THRESHOLD, 'target_signal'] = 0

df = df.merge(daily_sent, on='Date', how='left')
df['sentiment_score'] = df['sentiment_score'].fillna(0)
df['sentiment_score'] = df['sentiment_score'].shift(1).fillna(0)

FEATURE_COLS = ['sentiment_score','price_change_lag1','price_change_lag2',
                'volatility_5d','volume_delta','ma_ratio_5','ma_ratio_20']
df = df.dropna(subset=FEATURE_COLS+['target_signal']).reset_index(drop=True)

out_cols = ['Date','Close','sentiment_score','price_change_lag1',
            'price_change_lag2','volatility_5d','volume_delta',
            'ma_ratio_5','ma_ratio_20','target_signal']
features_df = df[out_cols].copy()
features_df.to_csv('/content/results/features.csv', index=False)

print(f"✅ Features saved: {len(features_df)} rows")
print("Target distribution:")
print(features_df['target_signal'].value_counts().rename({0:'DOWN',1:'HOLD',2:'UP'}))


## 9. Chronological Train-Test Split (No Lookahead Bias)

In [ ]:
TRAIN_END = pd.to_datetime(TRAIN_END_DATE)
train_df  = features_df[features_df['Date'] <= TRAIN_END].copy()
test_df   = features_df[features_df['Date'] >  TRAIN_END].copy()

print(f"Training : {len(train_df)} rows ({train_df['Date'].min().date()} to {train_df['Date'].max().date()})")
print(f"Test     : {len(test_df)} rows ({test_df['Date'].min().date()} to {test_df['Date'].max().date()})")
print(f"Test starts AFTER train: {test_df['Date'].min() > train_df['Date'].max()}")

X_train = train_df[FEATURE_COLS].values
y_train = train_df['target_signal'].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df['target_signal'].values

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)


## 10. Train Signal Classifier

In [ ]:
clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, C=1.0)
clf.fit(X_train_sc, y_train)

joblib.dump({'model': clf, 'scaler': scaler, 'feature_cols': FEATURE_COLS},
            '/content/models/signal_classifier.joblib')
print("✅ Signal classifier saved!")


## 11. Evaluate and Save Performance Metrics

In [ ]:
y_pred = clf.predict(X_test_sc)
report_dict = classification_report(y_test, y_pred,
    target_names=['DOWN','HOLD','UP'], output_dict=True)

perf_metrics = {
    cls: {
        'precision': round(report_dict[cls]['precision'], 4),
        'recall':    round(report_dict[cls]['recall'],    4),
        'f1-score':  round(report_dict[cls]['f1-score'],  4)
    } for cls in ['UP','DOWN','HOLD']
}

with open('/content/results/performance_metrics.json', 'w') as f:
    json.dump(perf_metrics, f, indent=2)

print(classification_report(y_test, y_pred, target_names=['DOWN','HOLD','UP']))
print("✅ Saved performance_metrics.json")


## 12. Backtest with Real Close Prices

In [ ]:
backtest_df = test_df.copy().reset_index(drop=True)
backtest_df['prediction'] = y_pred

INITIAL_CASH = 100_000.0
cash, shares = INITIAL_CASH, 0
portfolio_values = []

for _, row in backtest_df.iterrows():
    price  = float(row['Close'])
    signal = int(row['prediction'])
    if signal == 2 and cash >= price:
        shares_to_buy = int(cash // price)
        shares += shares_to_buy
        cash   -= shares_to_buy * price
    elif signal == 0 and shares > 0:
        cash  += shares * price
        shares = 0
    portfolio_values.append(cash + shares * price)

final_price  = float(backtest_df['Close'].iloc[-1])
final_value  = cash + shares * final_price
total_return = round((final_value - INITIAL_CASH) / INITIAL_CASH * 100, 4)

port_series = pd.Series(portfolio_values)
daily_rets  = port_series.pct_change().dropna()
sharpe      = round(float(daily_rets.mean() / (daily_rets.std() + 1e-10)) * math.sqrt(252), 4)

backtest_results = {
    'final_portfolio_value': round(final_value, 2),
    'total_return_pct':      total_return,
    'sharpe_ratio':          sharpe
}

with open('/content/results/backtest_results.json', 'w') as f:
    json.dump(backtest_results, f, indent=2)

print(json.dumps(backtest_results, indent=2))
print("✅ Saved backtest_results.json")


## 13. Feature Set Comparison

In [ ]:
def train_evaluate_f1(X_tr, X_te, y_tr, y_te):
    sc = StandardScaler()
    m  = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    m.fit(sc.fit_transform(X_tr), y_tr)
    preds = m.predict(sc.transform(X_te))
    rep   = classification_report(y_te, preds, target_names=['DOWN','HOLD','UP'],
                                   output_dict=True, zero_division=0)
    return round(rep['weighted avg']['f1-score'], 4)

SENT_COLS  = ['sentiment_score']
PRICE_COLS = ['price_change_lag1','price_change_lag2',
              'volatility_5d','volume_delta','ma_ratio_5','ma_ratio_20']

f1_sentiment = train_evaluate_f1(
    train_df[SENT_COLS].values, test_df[SENT_COLS].values, y_train, y_test)
f1_price = train_evaluate_f1(
    train_df[PRICE_COLS].values, test_df[PRICE_COLS].values, y_train, y_test)
f1_combined = round(report_dict['weighted avg']['f1-score'], 4)

comparison = {
    'sentiment_only_model_f1': f1_sentiment,
    'price_only_model_f1':     f1_price,
    'combined_model_f1':       f1_combined
}

with open('/content/results/feature_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print(json.dumps(comparison, indent=2))
print("✅ Saved feature_comparison.json")


## 14. Final Verification

In [ ]:
required_files = [
    '/content/data/finphrasebank.csv',
    '/content/data/stock_prices.csv',
    '/content/models/finetuned_distilbert/config.json',
    '/content/models/finetuned_distilbert/tokenizer_config.json',
    '/content/models/signal_classifier.joblib',
    '/content/results/features.csv',
    '/content/results/performance_metrics.json',
    '/content/results/backtest_results.json',
    '/content/results/feature_comparison.json',
]

print("Output Artifact Verification:")
print("-" * 55)
all_ok = True
for path in required_files:
    exists = os.path.isfile(path)
    size   = f"{os.path.getsize(path)/1024:,.1f} KB" if exists else "MISSING"
    status = "✅" if exists else "❌"
    short  = path.replace('/content/','')
    print(f"  {status}  {short:<50} {size}")
    all_ok = all_ok and exists

model_bin = (os.path.isfile('/content/models/finetuned_distilbert/pytorch_model.bin') or
             os.path.isfile('/content/models/finetuned_distilbert/model.safetensors'))
print(f"  {'✅' if model_bin else '❌'}  models/finetuned_distilbert/pytorch_model.bin")
print("-" * 55)
print("✅ All artifacts present! Ready to submit!" if (all_ok and model_bin)
      else "⚠ Some missing — rerun notebook from top.")


## 15. Download All Files

In [ ]:
import shutil
from google.colab import files

# Copy notebook itself
import shutil as sh
sh.copy('/content/notebooks/analysis.ipynb', '/content/data/../notebooks/analysis.ipynb') if os.path.exists('/content/notebooks/analysis.ipynb') else None

# Create zip
os.makedirs('/content/submission', exist_ok=True)
for folder in ['data', 'models', 'results', 'notebooks']:
    src = f'/content/{folder}'
    dst = f'/content/submission/{folder}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f"✅ {folder}/ added to zip")

shutil.make_archive('/content/final_submission', 'zip', '/content/submission')
size_mb = os.path.getsize('/content/final_submission.zip') / (1024*1024)
print(f"\n✅ ZIP ready! Size: {size_mb:.1f} MB")
print("Starting download...")
files.download('/content/final_submission.zip')
